In [ ]:
import boto3
import botocore
import functools
from IPython.core.display import display, HTML
from iterdub import iterdub as ib
from iterpop import iterpop as ip
import itertools as it
import json
import matplotlib
import matplotlib.pyplot as plt
import math
import numpy as np
import pandas as pd
from pandas.util import hash_pandas_object
import seaborn as sns
from teeplot import teeplot as tp


In [ ]:
from dishpylib.pyanalysis import calc_loglikelihoods_by_num_sets
from dishpylib.pyanalysis import count_hands_with_k_or_more_sets
from dishpylib.pyanalysis import count_hands_without_k_or_more_sets
from dishpylib.pyanalysis import estimate_interpolation_complexity
from dishpylib.pyanalysis import calc_loglikelihoods_over_set_sizes
from dishpylib.pyhelpers import get_env_context
from dishpylib.pyhelpers import get_git_revision_hash
from dishpylib.pyhelpers import make_timestamp
from dishpylib.pyhelpers import NumpyEncoder
from dishpylib.pyhelpers import preprocess_competition_fitnesses
from dishpylib.pyhelpers import print_runtime


In [ ]:
print_runtime()


In [ ]:
teeplot_subdir = "2025-11-30-genme-complexity-regression"


In [ ]:
s3_handle = boto3.resource(
    's3',
    region_name="us-east-2",
    config=botocore.config.Config(
        signature_version=botocore.UNSIGNED,
    ),
)
bucket_handle = s3_handle.Bucket("prq49")


control_competitions = bucket_handle.objects.filter(
    Prefix=f'endeavor=16/control-competitions/stage=2+what=collated/',
)


In [ ]:
control_dfs = [
    pd.read_csv(
        f's3://prq49/{control_competitions.key}',
    )
    for control_competitions in control_competitions
]


In [ ]:
pd.options.display.max_columns = None


In [ ]:
df = pd.concat(control_dfs, ignore_index=True)
df


In [ ]:
import pandas as pd
from scipy import stats

def fit_control_t_distns(control_df):

    na_rows = control_df['Fitness Differential'].isna()
    assert all( control_df[ na_rows ]['Population Extinct'] )
    control_df['Fitness Differential'].fillna(0, inplace=True,)

    res = []
    for series in control_df['Competition Series'].unique():

        series_df = control_df[ control_df['Competition Series'] == series ]

        # legacy data was mixed inside of the variant_df
        # wt_vs_wt_df = series_df.groupby('Competition Repro').filter(
        #     lambda x: (x['genome variation'] == 'master').all()
        # ).groupby('Competition Repro').first().reset_index()

        # fit a t distribution to the control data
        # df is degrees of freedom
        df, loc, scale = stats.t.fit( series_df['Fitness Differential'] )


        res.append({
            'Series' : series,
            'Fit Degrees of Freedom' : df,
            'Fit Loc' : loc,
            'Fit Scale' : scale,
        })

    return pd.DataFrame(res)


In [ ]:
import boto3
import botocore
import functools
import pandas as pd

from dishpylib.pyhelpers import fit_control_t_distns

@functools.lru_cache
def get_control_t_distns( bucket, endeavor, stint ):

    s3_handle = boto3.resource(
        's3',
        region_name="us-east-2",
        config=botocore.config.Config(
            signature_version=botocore.UNSIGNED,
        ),
    )
    bucket_handle = s3_handle.Bucket(bucket)

    control_competitions, = bucket_handle.objects.filter(
        Prefix=f'endeavor={endeavor}/control-competitions/stage=2+what=collated/stint={stint}/',
    )

    control_df = pd.read_csv(
        f's3://{bucket}/{control_competitions.key}',
    )
    print("mean update", control_df["Update"].mean())

    return fit_control_t_distns(control_df[
        control_df["Root ID"] == 1
    ].copy())


In [ ]:
import functools
from iterpop import iterpop as ip
from scipy import stats


def preprocess_competition_fitnesses(competitions_df, control_fits_df):
    # preprocess data
    @functools.lru_cache
    def h0_fit(series):
        return ip.popsingleton(
            control_fits_df[control_fits_df["Series"] == series].to_dict(
                orient="records",
            )
        )

    competitions_df["p"] = competitions_df.apply(
        lambda row: stats.t.cdf(
            row["Fitness Differential"],
            h0_fit(row["genome series"])["Fit Degrees of Freedom"],
            loc=h0_fit(row["genome series"])["Fit Loc"],
            scale=h0_fit(row["genome series"])["Fit Scale"],
        ),
        axis=1,
    )
    competitions_df["Is Less Fit"] = competitions_df["p"] < 1.0 / 50
    competitions_df["Is More Fit"] = competitions_df["p"] > (1.0 -  1.0 / 50)
    competitions_df["Is Neutral"] = ~(
        competitions_df["Is Less Fit"] | competitions_df["Is More Fit"]
    )
    competitions_df["Relative Fitness"] = competitions_df.apply(
        lambda row: (
            "Significantly Advantageous"
            if row["Is More Fit"]
            else (
                "Significantly Deleterious" if row["Is Less Fit"] else "Neutral"
            )
        ),
        axis=1,
    )

    return competitions_df


# get data


In [ ]:
pd.options.display.max_columns = None


In [ ]:
s3_handle = boto3.resource(
    's3',
    region_name="us-east-2",
    config=botocore.config.Config(
        signature_version=botocore.UNSIGNED,
    ),
)
bucket_handle = s3_handle.Bucket('prq49')

dfs = []
for stint in range(1, 101):
# for stint in (19,):
    print(stint)
    series_profiles, = bucket_handle.objects.filter(
        Prefix=f'endeavor=16/variant-competitions/stage=3+what=collated/stint={stint}/',
    )
    control_fits_df = get_control_t_distns('prq49', 16, stint)
    df = pd.read_csv(
        f's3://prq49/{series_profiles.key}',
        compression='xz',
    )
    print("mean update df", df["Update"].mean())

    df = df[df["Competition Series"] == 16005].copy()
    df["Stint"] = stint
    dfdigest = '{:x}'.format( hash_pandas_object( df ).sum() )
    print(dfdigest)
    df = preprocess_competition_fitnesses(df, control_fits_df)
    dfs.append(df)


In [ ]:
df = pd.concat(dfs)


In [ ]:
dfx = df[
    df["Root ID"] == 1
].groupby("Stint").agg(
    {
        "Is More Fit": "sum",
        "Is Less Fit": "sum",
        "Is Neutral": "sum",
    },
).reset_index(drop=False)
dfx


In [ ]:
sns.lineplot(
    data=dfx,
    markers=True,
    x="Stint",
    y="Is Less Fit",
)
sns.lineplot(
    data=dfx,
    markers=True,
    x="Stint",
    y="Is More Fit",
)
sns.lineplot(
    data=dfx,
    markers=True,
    x="Stint",
    y="Is Neutral",
)
plt.gca().set_ylabel("Num Sites")


In [ ]:
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

# Data
x = dfx["Stint"].values
y = dfx["Is Less Fit"].values

# Define breakpoint
c = 40

# Create piecewise regressor: slope changes after c
X = np.column_stack([
    x,
    np.where(x > c, x - c, 0)  # additional slope after breakpoint
])
X = sm.add_constant(X)

# Fit model
model = sm.OLS(y, X).fit()
print(model.summary())

# Predict for plotting
x_grid = np.linspace(x.min(), x.max(), 200)
X_grid = np.column_stack([
    x_grid,
    np.where(x_grid > c, x_grid - c, 0)
])
X_grid = sm.add_constant(X_grid)
y_grid = model.predict(X_grid)

# Plot
plt.scatter(x, y, alpha=0.6, label="Data")
plt.plot(x_grid, y_grid, color="red", label=f"Piecewise fit (break=40)")
plt.axvline(c, color="black", linestyle="--", label="Breakpoint = 40")
plt.xlabel("Stint")
plt.ylabel("Is Less Fit")
plt.legend()
plt.ylim(0, None)
plt.show()


In [ ]:
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

x = dfx["Stint"].values
y = dfx["Is Less Fit"].values

# -------------------------------
# Restricted model: single slope
# -------------------------------
X1 = sm.add_constant(x)
restricted = sm.OLS(y, X1).fit()

# -------------------------------
# Full model: breakpoint at 40
# -------------------------------
c = 40
X2 = np.column_stack([
    x,
    np.where(x > c, x - c, 0)  # slope change after breakpoint
])
X2 = sm.add_constant(X2)
full = sm.OLS(y, X2).fit()

# -------------------------------
# Compare models: nested F-test
# -------------------------------
f_test = full.compare_f_test(restricted)
print("F-test result:", f_test)


In [ ]:
import piecewise_regression

pw_fit = piecewise_regression.Fit(
    dfx["Stint"].to_list(),
    dfx["Is Less Fit"].to_list(),
    start_values=[40],
)
pw_fit.summary()


In [ ]:
pw_fit.plot_data(color="grey", s=20)
# Pass in standard matplotlib keywords to control any of the plots
pw_fit.plot_fit(
    color="red",
    linewidth=2,
    alpha=0.7,
    ls="--",
)
with tp.teed(
    pw_fit.plot_breakpoints,
    teeplot_subdir=teeplot_subdir,
    teeplot_outattrs={
        "x": "stint",
        "y": "genome_complexity",
    }
) as ax:
    pw_fit.plot_breakpoint_confidence_intervals()
    plt.xlabel("Stint")
    plt.ylabel("Genome\nComplexity")
    plt.gcf().set_size_inches(3, 1.8)

    # annotate breakpoint probability
    plt.text(77, 50, "adj $R^2=0.43$", fontsize=8)
    plt.text(40, 4, "Breakpoint\n$x=34$\n($p<1\mathrm{e}{-14}$)", fontsize=8)
    sns.despine(ax=ax)
